# Halal Stocks: ML Forecasting + Technical Analysis (daily & weekly RSI) + Financial Ratios

**What this notebook does**

1. Pulls **3 years** of daily prices from **Yahoo Finance** (`yfinance`) for the 50 US names in the *Halal Market Ledger* report of 8 Oct 2026, plus **SK Hynix**, **Rambus** and a basket of **cybersecurity** companies.
2. Computes **technical indicators on two time frames**:
   - **Daily:** RSI(14), SMA 10/20/50/100/200, EMA 12/26, MACD, Bollinger Bands, Stochastic, CCI, ADX/DI, OBV, ATR.
   - **Weekly** (daily bars resampled to Friday-close weeks): RSI(14), SMA 10/20/40-week, MACD.
   - Weekly values are attached to each day using only **completed weeks**, so nothing leaks from the future.
3. Pulls **financial ratios** (P/E, P/B, ROE, debt/equity, current ratio, margins, growth, ...) and builds a transparent 0-100 **fundamental score**.
4. Forecasts the **next-5-day return** with Ridge, Random Forest, Gradient Boosting and an ensemble, using **walk-forward validation** with a purge gap and naive/drift benchmarks.
5. **Validates everything**, not just the ML:
   - Does a **daily or weekly RSI** extreme actually precede different returns?
   - Do **moving-average regimes** and a combined **TA score** have any edge?
   - Which features does the model use, and does adding MA/RSI/weekly features **improve** it (ablation)?
6. Shows a **technical dashboard** (RSI daily vs weekly map, per-stock charts) and a final table combining **ML forecast + TA + fundamentals** ("confluence").

**Design choices that matter**

- We forecast **returns, not price levels** (prices are non-stationary).
- Financial ratios from Yahoo are a **current snapshot**, so they are **not** fed into the historical model (that would leak the present into the past). They are shown beside the forecast and used for the confluence check.
- Significance tests use non-overlapping samples and Benjamini-Hochberg correction across ~60 stocks.

**Run it:** `Runtime > Run all`. Expect ~10-15 minutes on a free Colab CPU (the ablation and feature-importance steps add a few minutes; set `RUN_ABLATION = False` to skip).

> Educational tool, not investment advice. The notebook is built to tell you honestly when the models and indicators find nothing.

In [ ]:
!pip -q install --upgrade yfinance

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from cycler import cycler
import yfinance as yf
from scipy import stats
from joblib import Parallel, delayed
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

# ----- chart style (validated categorical palette, light surface, thin marks, quiet grid) -----
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100",
         magenta="#e87ba4", green="#008300", violet="#6250d6", red="#e34948")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "lines.linewidth": 1.8, "font.size": 10, "figure.figsize": (11, 4.5),
    "axes.prop_cycle": cycler(color=[C[k] for k in ("blue", "orange", "aqua", "yellow", "magenta", "green", "violet", "red")]),
})

# ------------------------- CONFIG -------------------------
YEARS = 3                 # history window requested from Yahoo (the model sees only this window)
HORIZON = 5               # forecast horizon in trading days (1 = next day)
REFIT_EVERY = 42          # re-train the models every N trading days (smaller = slower, more faithful)
MIN_ROWS = 300            # skip tickers with less history than this
INIT_TRAIN_MAX = 252      # first training window (trading days)
RANDOM_STATE = 42
COST_BPS = 5              # backtest cost per position change, in basis points
BUY_THRESHOLD = 0.005     # forecast 5-day return must exceed +0.5% to be called a BUY
INCLUDE_TSX = False       # set True to also run the 50 TSX listings from the report
RUN_ABLATION = True       # compare feature sets (adds a few minutes); False to skip
RSI_OB, RSI_OS = 70, 30   # RSI overbought / oversold lines
MIN_FUND_SCORE = 50       # fundamental score (0-100) needed to count as "fundamentals OK"
CHART_TICKERS = ["NVDA", "MU", "RMBS", "000660.KS", "CRWD"]   # stocks that get a full technical chart

# 50 US names from the Halal Market Ledger report, 2026-10-08
US_HALAL = [
    "MU", "NVDA", "SNDK", "GOOGL", "NEM", "AVGO", "EOG", "AMAT", "SYK", "REGN",
    "ORCL", "QCOM", "ADBE", "GEV", "VRTX", "NKE", "LRCX", "COP", "MSFT", "TJX",
    "KLAC", "ASML", "HD", "ISRG", "TXN", "CRM", "CTAS", "CSCO", "CDNS", "ABT",
    "PG", "XOM", "JNJ", "LLY", "ANET", "LIN", "ITW", "AAPL", "ADI", "FTNT",
    "DHR", "TMO", "EMR", "AMD", "TSLA", "NOW", "MRK", "CRWD", "ABBV", "PANW",
]

# Added at your request (NOT in the report: check their Shariah status yourself)
EXTRAS = {
    "000660.KS": "SK Hynix (KRW, Korea Exchange)",
    "RMBS": "Rambus",
    "ZS": "Zscaler (cybersecurity)",
    "CHKP": "Check Point (cybersecurity)",
    "OKTA": "Okta (cybersecurity)",
    "S": "SentinelOne (cybersecurity)",
    "NET": "Cloudflare (cybersecurity/network)",
    "QLYS": "Qualys (cybersecurity)",
    "TENB": "Tenable (cybersecurity)",
}
# CRWD, PANW and FTNT (already above) are the other cybersecurity names.

TSX = [
    "K.TO", "AGI.TO", "ABX.TO", "LUN.TO", "STN.TO", "AEM.TO", "CLS.TO", "CVE.TO", "LNR.TO", "SU.TO",
    "GIB-A.TO", "GIL.TO", "KXS.TO", "NWC.TO", "TFII.TO", "TIH.TO", "CNQ.TO", "WPM.TO", "SAP.TO", "ATRL.TO",
    "CCL-B.TO", "CCO.TO", "NXE.TO", "FNV.TO", "CSU.TO", "IMO.TO", "MDA.TO", "TECK-B.TO", "WCP.TO", "CNR.TO",
    "RBA.TO", "CJT.TO", "LSPD.TO", "ARX.TO", "FM.TO", "CTC-A.TO", "CAE.TO", "DSG.TO", "OTEX.TO", "SHOP.TO",
    "CP.TO", "BB.TO", "MG.TO", "DOO.TO", "NFI.TO", "VET.TO", "WCN.TO", "TOU.TO", "GFL.TO", "EMP-A.TO",
]

UNIVERSE = list(dict.fromkeys(US_HALAL + list(EXTRAS) + (TSX if INCLUDE_TSX else [])))
print(f"{len(UNIVERSE)} tickers in the universe")

## 1. Download 3 years of daily data from Yahoo Finance

In [ ]:
END = pd.Timestamp.today().normalize()
START = END - pd.DateOffset(years=YEARS)
S, E = str(START.date()), str((END + pd.Timedelta(days=1)).date())
print(f"Window: {S} -> {END.date()}  ({YEARS} years)")

def _clean(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df[["Open", "High", "Low", "Close", "Volume"]].dropna(subset=["Close"])

wanted = UNIVERSE + ["SPY"]            # SPY = market context feature
raw = yf.download(wanted, start=S, end=E, auto_adjust=True, group_by="ticker",
                  progress=False, threads=True)
prices = {}
for t in wanted:
    try:
        prices[t] = _clean(raw[t])
    except Exception:
        pass

# retry anything that came back empty/short, one by one
for t in [t for t in wanted if t not in prices or len(prices[t]) < MIN_ROWS]:
    try:
        d = _clean(yf.download(t, start=S, end=E, auto_adjust=True, progress=False))
        if len(d):
            prices[t] = d
    except Exception:
        pass

assert "SPY" in prices, "Could not download SPY (needed for market features). Re-run the cell."
spy = prices["SPY"]["Close"]
stocks = {t: prices[t] for t in UNIVERSE if t in prices and len(prices[t]) >= MIN_ROWS}
dropped = [t for t in UNIVERSE if t not in stocks]
print(f"Usable tickers: {len(stocks)}")
if dropped:
    print("Skipped (no data / too short, e.g. recent IPO or delisted):", dropped)

# last 10 trading days of closing prices (native currency, split/dividend adjusted)
close = pd.DataFrame({t: d["Close"] for t, d in stocks.items()})
display(close.tail(10).round(2).T)

## 2. Technical indicators: moving averages, RSI (daily and weekly), MACD and more

All indicators are built from the 3-year window only. Definitions:

| Indicator | Daily | Weekly |
|---|---|---|
| **RSI(14)** (Wilder). Above 70 overbought, below 30 oversold | `rsi14` | `w_rsi14` (14 weekly bars) |
| **Moving averages** | SMA 10/20/50/100/200, EMA 12/26 | SMA 10/20/40 weeks |
| **MACD** (12, 26, 9) | `macd`, `macd_sig`, `macd_hist` | `w_macd`, `w_macd_hist` |
| Bollinger (20, 2), Stochastic %K (14), CCI (20), ADX/DI (14), OBV, ATR | yes | - |

**How weekly values avoid look-ahead:** daily bars are resampled to weeks ending Friday. A weekly value is attached to a day only once that week is **complete** (on Friday's close, or the next trading day after a Friday holiday). On Monday-Thursday the model sees the *previous* completed week. The dashboard additionally shows a **live** weekly RSI that includes the week in progress.

The **TA score** (-6 to +6) is a simple rule-based tally of six signals: price vs SMA50, SMA50 vs SMA200, daily MACD histogram, weekly MACD histogram, weekly price vs 20-week SMA, and daily RSI above/below 50. A score of +3 or more is "Bullish", -3 or less is "Bearish". RSI overbought/oversold is kept *out* of the score because it is a mean-reversion signal, tested separately in section 8.

In [ ]:
def wilder(x, n):
    """Wilder smoothing (EMA with alpha = 1/n); NaN until n observations."""
    return x.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

def rsi(c, n=14):
    d = c.diff()
    up, dn = wilder(d.clip(lower=0), n), wilder((-d.clip(upper=0)), n)
    out = 100 - 100 / (1 + up / dn)
    return out.where(dn != 0, 100.0).where(up.notna() & dn.notna())

def macd_parts(c, fast=12, slow=26, sig=9):
    m = c.ewm(span=fast, adjust=False).mean() - c.ewm(span=slow, adjust=False).mean()
    s = m.ewm(span=sig, adjust=False).mean()
    return m, s, m - s

def daily_panel(df):
    c, h, l, v = df["Close"], df["High"], df["Low"], df["Volume"]
    p = pd.DataFrame(index=df.index)
    p["close"] = c
    for n in (10, 20, 50, 100, 200):
        p[f"sma{n}"] = c.rolling(n).mean()
    p["ema12"], p["ema26"] = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    p["rsi14"] = rsi(c, 14)
    p["macd"], p["macd_sig"], p["macd_hist"] = macd_parts(c)
    m20, s20 = c.rolling(20).mean(), c.rolling(20).std()
    p["bb_up"], p["bb_lo"] = m20 + 2 * s20, m20 - 2 * s20
    p["bb_pctb"] = (c - p["bb_lo"]) / (p["bb_up"] - p["bb_lo"])
    p["bb_width"] = (p["bb_up"] - p["bb_lo"]) / m20
    ll, hh = l.rolling(14).min(), h.rolling(14).max()
    p["stoch_k"] = 100 * (c - ll) / (hh - ll).replace(0, np.nan)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = wilder(tr, 14)
    p["atr14"] = atr / c
    up, dn = h.diff(), -l.diff()
    pdm = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=df.index)
    mdm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=df.index)
    pdi, mdi = 100 * wilder(pdm, 14) / atr, 100 * wilder(mdm, 14) / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    p["adx14"], p["di_diff"] = wilder(dx, 14), pdi - mdi
    tp = (h + l + c) / 3
    mad = tp.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    p["cci20"] = (tp - tp.rolling(20).mean()) / (0.015 * mad.replace(0, np.nan))
    obv = (np.sign(c.diff()).fillna(0) * v).cumsum()
    p["obv20"] = (obv - obv.shift(20)) / (v.rolling(20).sum() + 1)
    p["vol_z"] = np.log((v + 1) / (v.rolling(20).mean() + 1))
    return p

def weekly_panel(df):
    """Weekly bars ending Friday. The LAST row is the week in progress (live)."""
    w = df.resample("W-FRI").agg({"Open": "first", "High": "max", "Low": "min",
                                  "Close": "last", "Volume": "sum"}).dropna(subset=["Close"])
    c = w["Close"]
    q = pd.DataFrame(index=w.index)
    q["w_close"] = c
    for n in (10, 20, 40):
        q[f"w_sma{n}"] = c.rolling(n).mean()
    q["w_rsi14"] = rsi(c, 14)
    q["w_rsi_chg"] = q["w_rsi14"].diff()
    m, s, hst = macd_parts(c)
    settled = np.arange(len(c)) >= 26                    # weekly EMAs are not settled before ~26 weeks
    q["w_macd"], q["w_macd_sig"], q["w_macd_hist"] = m.where(settled), s.where(settled), hst.where(settled)
    return q

def sgn(x):
    return np.sign(x).fillna(0)

def ta_score(p):
    return (sgn(p["close"] - p["sma50"]) + sgn(p["sma50"] - p["sma200"]) + sgn(p["macd_hist"])
            + sgn(p["w_macd_hist"]) + sgn(p["w_close"] - p["w_sma20"]) + sgn(p["rsi14"] - 50))

def full_panel(df):
    d, wk = daily_panel(df), weekly_panel(df)
    aligned = wk.reindex(d.index, method="ffill")        # last COMPLETED week as of each day's close
    p = pd.concat([d, aligned], axis=1)
    p["rsi_d_minus_w"] = p["rsi14"] - p["w_rsi14"]
    p["ta_score"] = ta_score(p)
    return p

panels = {t: full_panel(d) for t, d in stocks.items()}
t0 = next(iter(panels))
print(f"{t0}: {panels[t0].shape[1]} indicator columns, {len(panels[t0])} daily rows")
display(panels[t0][["close", "sma20", "sma50", "sma200", "rsi14", "w_rsi14", "macd_hist", "w_macd_hist", "ta_score"]].tail(6).round(2))

## 3. Features and target

All features use only data available at the close of day *t*. The target is the **log return from *t* to *t+5***.

Feature groups (used later for the ablation test):
- **Base (22):** returns, volatility, SMA 10/20/50 distance, RSI14, MACD histogram, Bollinger %B, ATR, volume z-score, distance to 52-week high, market (SPY) returns.
- **MA / daily TA (10):** SMA100 distance, SMA50 vs SMA100, EMA12 vs EMA26, RSI 5-day change, Stochastic, CCI, ADX, DI difference, OBV, Bollinger width.
- **Weekly (6):** weekly RSI, weekly RSI change, price vs 10- and 20-week SMA, weekly MACD histogram, daily-minus-weekly RSI.

In [ ]:
def build_dataset(p, spy_close, horizon=HORIZON):
    c = p["close"]
    lc = np.log(c)
    r = lc.diff()
    f = pd.DataFrame(index=p.index)
    # ---- base set
    for k in (1, 2, 3, 5, 10, 21, 63):
        f[f"ret_{k}"] = lc.diff(k)
    for k in (10, 21, 63):
        f[f"vol_{k}"] = r.rolling(k).std()
    for n in (10, 20, 50):
        f[f"sma_{n}"] = c / p[f"sma{n}"] - 1
    f["sma10_50"] = p["sma10"] / p["sma50"] - 1
    f["rsi14"] = p["rsi14"]
    f["macd_hist"] = p["macd_hist"] / c
    f["bb_pctb"] = p["bb_pctb"]
    f["atr14"] = p["atr14"]
    f["vol_z"] = p["vol_z"]
    f["dist_high"] = c / c.rolling(252, min_periods=60).max() - 1
    mk = np.log(spy_close).reindex(p.index).ffill()
    f["mkt_ret_5"], f["mkt_ret_21"] = mk.diff(5), mk.diff(21)
    # ---- moving averages / daily technical analysis
    f["sma_100"] = c / p["sma100"] - 1
    f["sma50_100"] = p["sma50"] / p["sma100"] - 1
    f["ema12_26"] = p["ema12"] / p["ema26"] - 1
    f["rsi_chg5"] = p["rsi14"].diff(5)
    for col in ("stoch_k", "cci20", "adx14", "di_diff", "obv20", "bb_width"):
        f[col] = p[col]
    # ---- weekly
    f["w_rsi14"], f["w_rsi_chg"] = p["w_rsi14"], p["w_rsi_chg"]
    f["w_sma10"] = p["w_close"] / p["w_sma10"] - 1
    f["w_sma20"] = p["w_close"] / p["w_sma20"] - 1
    f["w_macd_hist"] = p["w_macd_hist"] / p["w_close"]
    f["rsi_d_minus_w"] = p["rsi_d_minus_w"]
    f = f.replace([np.inf, -np.inf], np.nan)
    y = (lc.shift(-horizon) - lc).rename("y")        # forward log return, NaN for the last `horizon` rows
    return f, y, c

datasets = {t: build_dataset(panels[t], spy) for t in panels}

BASE_COLS = ([f"ret_{k}" for k in (1, 2, 3, 5, 10, 21, 63)] + [f"vol_{k}" for k in (10, 21, 63)]
             + ["sma_10", "sma_20", "sma_50", "sma10_50", "rsi14", "macd_hist", "bb_pctb", "atr14",
                "vol_z", "dist_high", "mkt_ret_5", "mkt_ret_21"])
TA_COLS = ["sma_100", "sma50_100", "ema12_26", "rsi_chg5", "stoch_k", "cci20", "adx14", "di_diff", "obv20", "bb_width"]
WEEKLY_COLS = ["w_rsi14", "w_rsi_chg", "w_sma10", "w_sma20", "w_macd_hist", "rsi_d_minus_w"]
ALL_COLS = BASE_COLS + TA_COLS + WEEKLY_COLS
assert set(ALL_COLS) == set(datasets[t0][0].columns), "feature list mismatch"
print(f"{len(BASE_COLS)} base + {len(TA_COLS)} MA/TA + {len(WEEKLY_COLS)} weekly = {len(ALL_COLS)} features")

## 4. Financial ratios (fundamentals)

Pulled from Yahoo Finance for each stock. Two honest caveats:

- Yahoo gives only the **latest** ratios, not a point-in-time history. Using them as model features over the last 3 years would leak today's information into the past, so they are **not** used for training. They sit beside the forecast and feed the confluence check at the end.
- `D/E` is converted to a ratio (Yahoo reports it in percent). This is **book** D/E, not the AAOIFI debt-to-market-cap Shariah test.

The **fundamental score (0-100)** is my own simplified version (not the report's 105-point model): five components of 20 points each, following the report's colour bands.

| Component | Full marks | Zero |
|---|---|---|
| Valuation (trailing P/E) | P/E <= 20 | P/E >= 150 (loss-making scores 2) |
| ROE | >= 30% | <= 0% |
| Leverage (debt/equity) | <= 0.5 | >= 3.0 |
| Liquidity (current ratio) | >= 2.0 | <= 0.5 |
| Growth (revenue growth) | >= 20% | <= -10% |

Missing inputs get a neutral mid-score; if fewer than 3 of 5 inputs exist the score is blank.

In [ ]:
def fnum(x):
    try:
        x = float(x)
        return x if np.isfinite(x) else np.nan
    except Exception:
        return np.nan

def fetch_info(t):
    for attempt in range(3):
        try:
            info = yf.Ticker(t).info
            if isinstance(info, dict) and len(info) > 5:
                return t, info
        except Exception:
            pass
        time.sleep(1.0 + attempt)
    return t, {}

with ThreadPoolExecutor(max_workers=6) as ex:
    infos = dict(ex.map(fetch_info, list(stocks)))

rows = []
for t, info in infos.items():
    last = float(stocks[t]["Close"].iloc[-1])
    de = fnum(info.get("debtToEquity"))
    tgt = fnum(info.get("targetMeanPrice"))
    rows.append(dict(
        ticker=t, pe=fnum(info.get("trailingPE")), fwd_pe=fnum(info.get("forwardPE")), pb=fnum(info.get("priceToBook")),
        ps=fnum(info.get("priceToSalesTrailing12Months")), ev_ebitda=fnum(info.get("enterpriseToEbitda")),
        roe=fnum(info.get("returnOnEquity")), roa=fnum(info.get("returnOnAssets")),
        net_margin=fnum(info.get("profitMargins")), op_margin=fnum(info.get("operatingMargins")),
        rev_growth=fnum(info.get("revenueGrowth")), earn_growth=fnum(info.get("earningsGrowth")),
        de=de / 100 if np.isfinite(de) else np.nan,
        current=fnum(info.get("currentRatio")), quick=fnum(info.get("quickRatio")),
        fcf_b=fnum(info.get("freeCashflow")) / 1e9, mcap_b=fnum(info.get("marketCap")) / 1e9,
        beta=fnum(info.get("beta")),
        target_upside=(tgt / last - 1) if np.isfinite(tgt) and last > 0 else np.nan))
fund = pd.DataFrame(rows).set_index("ticker")

def _interp(x, xp, fp, neutral):
    return neutral if not np.isfinite(x) else float(np.interp(x, xp, fp))

def fundamental_score(r):
    if sum(np.isfinite(r[k]) for k in ("pe", "roe", "de", "current", "rev_growth")) < 3:
        return np.nan
    pe = r["pe"]
    if not np.isfinite(pe):
        val = 6.0
    elif pe <= 0:
        val = 2.0
    else:
        val = float(np.interp(pe, [0, 20, 50, 150], [20, 20, 8, 0]))
    return (val + _interp(r["roe"], [0, 0.10, 0.30], [0, 10, 20], 8.0)
            + _interp(r["de"], [0.5, 1.5, 3.0], [20, 8, 0], 12.0)
            + _interp(r["current"], [0.5, 1.0, 2.0], [0, 8, 20], 8.0)
            + _interp(r["rev_growth"], [-0.10, 0.0, 0.20], [0, 8, 20], 8.0))

fund["fund_score"] = fund.apply(fundamental_score, axis=1)
fund["fund_view"] = pd.cut(fund["fund_score"], [-1, 50, 65, 101], labels=["Weak", "Fair", "Strong"]).astype(object)
print(f"Ratios retrieved for {int(fund['fund_score'].notna().sum())} of {len(fund)} tickers"
      f" ({int((fund['fund_score'].isna()).sum())} with too little data; Yahoo sometimes throttles, re-run this cell to retry)")
show = fund.sort_values("fund_score", ascending=False)
display(show.round(3))

if show["fund_score"].notna().any():
    s = show["fund_score"].dropna()[::-1]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.22 * len(s))))
    ax.barh(s.index, s.values, color=C["blue"], height=0.65)
    ax.axvline(MIN_FUND_SCORE, color=MUTED, ls="--", lw=1)
    ax.set_title(f"Fundamental score (0-100); dashed line = {MIN_FUND_SCORE}, the 'fundamentals OK' threshold")
    ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8)
    plt.show()

## 5. Models and walk-forward validation

Models: **Ridge** (linear, heavily regularised), **RandomForest**, **HistGradientBoosting**, and an **Ensemble** (average). Benchmarks: **Naive** (predict 0% return = price unchanged) and **Drift** (the training-set average return).

At every refit date, models train only on rows whose 5-day target was already fully known (purge gap), then forecast the next `REFIT_EVERY` days. Repeating this across the out-of-sample period gives forecasts that mimic live use. Every model uses all 38 features.

In [ ]:
ML_NAMES = ("Ridge", "RandomForest", "HistGB")
ML_MODELS = ["Ridge", "RandomForest", "HistGB", "Ensemble"]
BENCH = ["Naive", "Drift"]
ALL_MODELS = BENCH + ML_MODELS

def make_models():
    return {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=50.0)),
        "RandomForest": RandomForestRegressor(n_estimators=100, min_samples_leaf=10, max_features=0.5,
                                              n_jobs=1, random_state=RANDOM_STATE),
        "HistGB": HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=100,
                                                min_samples_leaf=20, l2_regularization=1.0,
                                                random_state=RANDOM_STATE),
    }

def fit_predict_all(Xtr, ytr, Xte, names=ML_NAMES):
    lo, hi = ytr.quantile([0.01, 0.99])
    ytr_c = ytr.clip(lo, hi)                           # tame outliers in the training target
    out = {"Naive": np.zeros(len(Xte)), "Drift": np.full(len(Xte), float(ytr.mean()))}
    models = make_models()
    for name in names:
        models[name].fit(Xtr, ytr_c)
        out[name] = models[name].predict(Xte)
    out["Ensemble"] = np.mean([out[n] for n in names], axis=0)
    return out

def walk_forward(feat, y, close_s, cols=None, names=ML_NAMES, horizon=HORIZON, refit=REFIT_EVERY):
    cols = list(cols) if cols is not None else list(feat.columns)
    data = feat.join(y).dropna(subset=list(feat.columns))     # same rows for every feature subset
    n = len(data)
    init = min(INIT_TRAIN_MAX, int(0.45 * n))
    X, yy = data[cols], data["y"]
    parts = []
    for start in range(init, n, refit):
        train_end = start - horizon + 1                # rows i <= start-horizon have a fully known target
        Xtr, ytr = X.iloc[:train_end], yy.iloc[:train_end]
        Xte = X.iloc[start:start + refit]
        preds = fit_predict_all(Xtr, ytr, Xte, names)
        parts.append(pd.DataFrame(preds, index=Xte.index))
    res = pd.concat(parts)
    res["y"] = yy.reindex(res.index)
    res["close"] = close_s.reindex(res.index)
    res["ret_next"] = np.log(close_s.shift(-1) / close_s).reindex(res.index)   # 1-day-ahead return for backtest
    return res

t_start = time.time()
tickers = list(datasets)
out = Parallel(n_jobs=-1, verbose=5)(
    delayed(walk_forward)(datasets[t][0], datasets[t][1], datasets[t][2], ALL_COLS) for t in tickers)
results = dict(zip(tickers, out))
print(f"\nWalk-forward finished for {len(results)} tickers in {time.time() - t_start:.0f}s")

## 6. Validation metrics

For each stock and model we report:

- **rmse_vs_naive**: forecast error divided by the naive no-change error. Below 1 means the model beats "price stays the same".
- **oos_r2_vs_drift**: out-of-sample R-squared against the historical-average forecast. Positive means real skill.
- **dir_acc** vs **always_up_acc**: directional hit rate vs. simply always guessing "up". **dir_pval** is a one-sided binomial test on **non-overlapping** forecasts.
- **ic**: Spearman rank correlation between forecast and realised return (information coefficient).
- **dir_qval**: Benjamini-Hochberg adjusted p-value across all tickers (guards against lucky stocks).

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def bh_qvalues(p):
    p = np.asarray(p, float)
    n = len(p)
    order = np.argsort(p)
    q = np.empty(n)
    prev = 1.0
    for rank, idx in zip(range(n, 0, -1), order[::-1]):
        prev = min(prev, p[idx] * n / rank)
        q[idx] = prev
    return q

def evaluate(ticker, res):
    ev = res.dropna(subset=["y"])
    no = ev.iloc[::HORIZON]                            # non-overlapping forecasts for significance tests
    base_up, base_up_no = (ev["y"] > 0).mean(), (no["y"] > 0).mean()
    r_naive = rmse(ev["y"], ev["Naive"])
    rows = []
    for m in ALL_MODELS:
        p = ev[m]
        row = dict(ticker=ticker, model=m, n_test=len(ev),
                   rmse=rmse(ev["y"], p), rmse_vs_naive=rmse(ev["y"], p) / r_naive,
                   oos_r2_vs_drift=1 - ((ev["y"] - p) ** 2).sum() / ((ev["y"] - ev["Drift"]) ** 2).sum(),
                   price_mape_pct=float(np.mean(np.abs(np.exp(p) - np.exp(ev["y"])) / np.exp(ev["y"])) * 100),
                   always_up_acc=base_up, dir_acc=np.nan, dir_pval=np.nan, ic=np.nan, ic_p=np.nan)
        if m != "Naive":
            row["dir_acc"] = ((p > 0) == (ev["y"] > 0)).mean()
            k = int(((no[m] > 0) == (no["y"] > 0)).sum())
            row["dir_pval"] = stats.binomtest(k, len(no), min(max(base_up_no, 1e-6), 1 - 1e-6),
                                              alternative="greater").pvalue
            if p.std() > 0:
                row["ic"] = stats.spearmanr(p, ev["y"])[0]
                row["ic_p"] = stats.spearmanr(no[m], no["y"])[1]
        rows.append(row)
    return pd.DataFrame(rows)

metrics = pd.concat([evaluate(t, r) for t, r in results.items()], ignore_index=True)
metrics["dir_qval"] = np.nan
for m in metrics["model"].unique():
    idx = metrics.index[(metrics["model"] == m) & metrics["dir_pval"].notna()]
    if len(idx):
        metrics.loc[idx, "dir_qval"] = bh_qvalues(metrics.loc[idx, "dir_pval"].values)

def agg_table(m):
    rows = []
    for model, g in m.groupby("model"):
        row = dict(model=model, tickers=len(g),
                   mean_rmse_ratio=g["rmse_vs_naive"].mean(),
                   pct_beat_naive=(g["rmse_vs_naive"] < 1).mean() * 100,
                   mean_oos_r2=g["oos_r2_vs_drift"].mean(),
                   mean_dir_acc=g["dir_acc"].mean(), mean_always_up=g["always_up_acc"].mean(),
                   pct_dir_sig_5pct=(g["dir_pval"] < 0.05).mean() * 100,
                   pct_dir_sig_after_BH=(g["dir_qval"] < 0.10).mean() * 100,
                   mean_ic=g["ic"].mean(), mean_price_mape_pct=g["price_mape_pct"].mean())
        try:
            row["wilcoxon_p_beats_naive"] = stats.wilcoxon(g["rmse_vs_naive"] - 1, alternative="less").pvalue
        except Exception:
            row["wilcoxon_p_beats_naive"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("model").loc[ALL_MODELS]

agg = agg_table(metrics)
display(agg.round(4))

cand = agg.loc[ML_MODELS]
BEST = cand["mean_rmse_ratio"].idxmin()
print(f"\nBest approach by out-of-sample error vs naive: {BEST}  (mean RMSE ratio {cand.loc[BEST, 'mean_rmse_ratio']:.4f})")
if cand.loc[BEST, "mean_rmse_ratio"] >= 1:
    print("No ML model beat the naive no-change forecast on average. Treat any BUY list below as weak evidence.")
else:
    print(f"{BEST} beats the naive forecast on {cand.loc[BEST, 'pct_beat_naive']:.0f}% of tickers.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
agg.loc[ML_MODELS, "mean_rmse_ratio"].plot.bar(ax=ax[0], color=C["blue"], rot=0)
ax[0].axhline(1, color=MUTED, ls="--", lw=1); ax[0].set_title("Mean forecast error vs naive (below 1 = better)"); ax[0].set_xlabel("")
ax[0].set_ylim(bottom=min(0.9, agg.loc[ML_MODELS, "mean_rmse_ratio"].min() - 0.02))

pivot = metrics[metrics.model == BEST].set_index("ticker")
ax[1].hist((pivot["dir_acc"] - pivot["always_up_acc"]) * 100, bins=20, color=C["blue"], edgecolor=SURFACE)
ax[1].axvline(0, color=MUTED, ls="--", lw=1)
ax[1].set_title(f"{BEST}: direction hit rate minus always-up (pct pts)")

ax[2].hist(pivot["ic"].dropna(), bins=20, color=C["blue"], edgecolor=SURFACE)
ax[2].axvline(0, color=MUTED, ls="--", lw=1); ax[2].set_title(f"{BEST}: forecast-vs-actual rank correlation per stock")
plt.tight_layout(); plt.show()

## 7. Validating the forecast trend

Three direct checks of whether the *trend the model predicts* shows up in reality.

In [ ]:
# 7a. Calibration: do higher forecasts precede higher realised returns? (non-overlapping forecasts, ranked within each stock)
frames = []
for t, res in results.items():
    ev = res.dropna(subset=["y"]).iloc[::HORIZON]
    if len(ev) < 25:
        continue
    q = pd.qcut(ev[BEST].rank(method="first"), 5, labels=False)
    frames.append(pd.DataFrame({"t": t, "q": q.values, "y": ev["y"].values}))
qd = pd.concat(frames)
by_q = qd.groupby("q")["y"].agg(["mean", "count"])
by_q["mean_pct"] = by_q["mean"] * 100
spread = qd.groupby(["t", "q"])["y"].mean().unstack()
spread = (spread[4] - spread[0]).dropna()
tstat, pval = stats.ttest_1samp(spread, 0)

display(by_q.rename(index={0: "Q1 (lowest forecast)", 4: "Q5 (highest forecast)"}).round(4))
print(f"Top-minus-bottom quintile spread: {spread.mean()*100:.2f}% per {HORIZON}d, "
      f"t = {tstat:.2f}, p = {pval:.3f}  ->  {'monotonic edge present' if (pval < 0.05 and spread.mean() > 0) else 'no reliable edge'}")
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(by_q.index.astype(str), by_q["mean_pct"], color=C["blue"], width=0.6)
ax.axhline(0, color=AXIS, lw=1)
ax.set_title(f"Average realised {HORIZON}-day return by forecast quintile ({BEST})"); ax.set_ylabel("%")
ax.set_xlabel("forecast quintile (0 = lowest forecast, 4 = highest)"); ax.grid(axis="x", visible=False)
plt.show()

In [ ]:
# 7b. Stability over time: pooled directional accuracy vs always-up, per quarter
allev = pd.concat([r.dropna(subset=["y"]).assign(t=t) for t, r in results.items()])
allev["qtr"] = allev.index.to_period("Q")
stab = allev.groupby("qtr").apply(lambda g: pd.Series({
    "model_hit_rate": ((g[BEST] > 0) == (g["y"] > 0)).mean(),
    "always_up_rate": (g["y"] > 0).mean(),
    "n": len(g)}))
stab["edge_pct_pts"] = (stab["model_hit_rate"] - stab["always_up_rate"]) * 100
display(stab.round(3))
print(f"Quarters where {BEST} beat always-up: {(stab['edge_pct_pts'] > 0).sum()} of {len(stab)}")
ax = stab[["model_hit_rate", "always_up_rate"]].plot(marker="o", ms=5, title="Directional hit rate by quarter")
ax.legend(["Model", "Always guess up"]); plt.show()

In [ ]:
# 7c. Cost-aware backtest: long when forecast > threshold, else cash. Equal-weight across stocks.
def backtest(res, model, thr=BUY_THRESHOLD, cost_bps=COST_BPS):
    d = res.dropna(subset=["ret_next"])
    pos = (d[model] > thr).astype(float)               # decided at close t, earns return t -> t+1 (no look-ahead)
    turn = pos.diff().abs().fillna(pos.abs())
    return pd.DataFrame({"strategy": pos * d["ret_next"] - turn * cost_bps / 1e4,
                         "buyhold": d["ret_next"], "pos": pos})

bts = {t: backtest(r, BEST) for t, r in results.items()}
strat = pd.concat({t: b["strategy"] for t, b in bts.items()}, axis=1).mean(axis=1)
bh = pd.concat({t: b["buyhold"] for t, b in bts.items()}, axis=1).mean(axis=1)

def perf(s):
    cum = np.exp(s.cumsum())
    ann, vol = s.mean() * 252, s.std() * np.sqrt(252)
    return pd.Series({"total_return_%": (cum.iloc[-1] - 1) * 100, "ann_return_%": ann * 100, "ann_vol_%": vol * 100,
                      "sharpe": ann / vol if vol > 0 else np.nan, "max_drawdown_%": (cum / cum.cummax() - 1).min() * 100})

perf_tbl = pd.DataFrame({f"{BEST} long/cash": perf(strat), "Buy & hold (equal weight)": perf(bh)}).T
display(perf_tbl.round(2))
ax = np.exp(pd.DataFrame({f"{BEST} long/cash": strat, "Buy & hold": bh}).cumsum()).plot(
    title="Out-of-sample growth of 1.0 (equal weight, after costs)")
plt.show()

per_t = pd.DataFrame({t: {"strategy_sharpe": perf(b["strategy"])["sharpe"], "buyhold_sharpe": perf(b["buyhold"])["sharpe"],
                          "time_invested_%": b["pos"].mean() * 100} for t, b in bts.items()}).T
print(f"Strategy Sharpe beat buy-and-hold on {(per_t['strategy_sharpe'] > per_t['buyhold_sharpe']).mean()*100:.0f}% of stocks")

In [ ]:
# Example: forecast vs. actual price for one stock (forecast made HORIZON days earlier)
TICKER_TO_PLOT = "NVDA" if "NVDA" in results else next(iter(results))     # change me
res = results[TICKER_TO_PLOT]
fc = (res["close"] * np.exp(res[BEST])).shift(HORIZON)
ax = pd.DataFrame({"Actual close": res["close"], f"{BEST} forecast (made {HORIZON} days earlier)": fc}).plot(figsize=(12, 4.5))
ax.set_title(f"{TICKER_TO_PLOT}: actual vs {HORIZON}-day-ahead forecast (out-of-sample)"); plt.show()

## 8. Does technical analysis actually work here?

Each test asks: *when the indicator is in state X, are the next 5-day returns different from that stock's normal return?*

- **avg_excess_5d_%**: average 5-day return in that state minus the stock's overall average (so a stock that simply went up a lot does not flatter every bucket).
- **p_tickers**: t-test across stocks (each stock counts once).
- **p_dates**: t-test across **non-overlapping dates** after averaging all stocks on each date. This is the stricter one, because stocks tend to move together (a market sell-off makes many stocks "oversold" at once and they rebound together).
- Treat a result as real only if **both** p-values are small *and* the pattern is consistent with common sense. With ~5 tests per indicator, a lone p of 0.04 is weak evidence.

In [ ]:
def bucket_validation(label_fn, order, title):
    frames = []
    for t, p in panels.items():
        lab = label_fn(p)
        d = pd.concat([lab.rename("bucket"), datasets[t][1]], axis=1).dropna()
        if len(d) < 80:
            continue
        d["excess"] = d["y"] - d["y"].mean()
        d["ticker"] = t
        frames.append(d)
    pan = pd.concat(frames)
    pan.index.name = "date"
    pan = pan.reset_index()

    per_t = pan.groupby(["ticker", "bucket"])["excess"].agg(["mean", "count"]).reset_index()
    per_t = per_t[per_t["count"] >= 5]
    dates = np.sort(pan["date"].unique())[::HORIZON]                      # non-overlapping dates
    per_d = pan[pan["date"].isin(dates)].groupby(["date", "bucket"])["excess"].mean().reset_index()

    rows = []
    for b in order:
        a = per_t.loc[per_t["bucket"] == b, "mean"]
        dd = per_d.loc[per_d["bucket"] == b, "excess"]
        if len(a) < 5:
            continue
        sub = pan[pan["bucket"] == b]
        rows.append({"state": b, "observations": len(sub), "stocks": len(a),
                     "avg_excess_5d_%": a.mean() * 100, "se_%": a.std() / np.sqrt(len(a)) * 100,
                     "hit_rate_up_%": (sub["y"] > 0).mean() * 100,
                     "p_tickers": stats.ttest_1samp(a, 0).pvalue,
                     "p_dates": stats.ttest_1samp(dd, 0).pvalue if len(dd) >= 10 else np.nan})
    tbl = pd.DataFrame(rows).set_index("state")
    print(f"{title}\n(overall: {(pan['y'] > 0).mean()*100:.1f}% of {HORIZON}-day windows were up)")
    display(tbl.drop(columns=["se_%"]).round(3))
    if len(tbl):
        fig, ax = plt.subplots(figsize=(8, 0.55 * len(tbl) + 1.6))
        pos = np.arange(len(tbl))
        ax.barh(pos, tbl["avg_excess_5d_%"], xerr=tbl["se_%"], color=C["blue"], height=0.6,
                error_kw=dict(ecolor=INK2, lw=1, capsize=3))
        ax.axvline(0, color=AXIS, lw=1)
        ax.set_yticks(pos); ax.set_yticklabels(tbl.index); ax.invert_yaxis()
        ax.set_xlabel(f"average {HORIZON}-day return minus the stock's normal (%), bars = +/-1 standard error")
        ax.set_title(title); ax.grid(axis="y", visible=False)
        plt.tight_layout(); plt.show()
    return tbl

RSI_LABELS = ["<30 oversold", "30-45", "45-55", "55-70", ">70 overbought"]

def lab_rsi_daily(p):
    return pd.cut(p["rsi14"], [0, 30, 45, 55, 70, 100], labels=RSI_LABELS, include_lowest=True).astype(object)

def lab_rsi_weekly(p):
    return pd.cut(p["w_rsi14"], [0, 30, 45, 55, 70, 100], labels=RSI_LABELS, include_lowest=True).astype(object)

JOINT = ["Both overbought (daily & weekly >= 70)", "Daily overbought, weekly <= 50", "Other",
         "Daily oversold, weekly >= 50 (dip in uptrend)", "Both oversold (daily & weekly <= 30)"]

def lab_rsi_joint(p):
    d, w = p["rsi14"], p["w_rsi14"]
    ok = d.notna() & w.notna()
    out = pd.Series(np.nan, index=p.index, dtype=object)
    out[ok] = JOINT[2]
    out[ok & (d >= 70) & (w >= 70)] = JOINT[0]
    out[ok & (d >= 70) & (w <= 50)] = JOINT[1]
    out[ok & (d <= 30) & (w >= 50)] = JOINT[3]
    out[ok & (d <= 30) & (w <= 30)] = JOINT[4]
    return out

MA_STATES = ["Uptrend (price > SMA50 > SMA200)", "Pullback (price < SMA50 > SMA200)",
             "Recovery (price > SMA50 < SMA200)", "Downtrend (price < SMA50 < SMA200)"]

def lab_ma(p):
    c, s50, s200 = p["close"], p["sma50"], p["sma200"]
    ok = s200.notna()
    out = pd.Series(np.nan, index=p.index, dtype=object)
    out[ok & (c > s50) & (s50 > s200)] = MA_STATES[0]
    out[ok & (c < s50) & (s50 > s200)] = MA_STATES[1]
    out[ok & (c > s50) & (s50 < s200)] = MA_STATES[2]
    out[ok & (c < s50) & (s50 < s200)] = MA_STATES[3]
    return out

TA_STATES = ["Bearish (score <= -3)", "Neutral (-2 to +2)", "Bullish (score >= +3)"]

def lab_ta(p):
    ok = p[["sma200", "w_sma20", "w_macd_hist"]].notna().all(axis=1)
    s = p["ta_score"]
    out = pd.Series(np.nan, index=p.index, dtype=object)
    out[ok & (s <= -3)] = TA_STATES[0]
    out[ok & (s > -3) & (s < 3)] = TA_STATES[1]
    out[ok & (s >= 3)] = TA_STATES[2]
    return out

tbl_rsi_d = bucket_validation(lab_rsi_daily, RSI_LABELS, "Daily RSI(14) state and the next 5-day return")

In [ ]:
tbl_rsi_w = bucket_validation(lab_rsi_weekly, RSI_LABELS, "Weekly RSI(14) state and the next 5-day return")

In [ ]:
tbl_rsi_j = bucket_validation(lab_rsi_joint, JOINT, "Daily + weekly RSI together")

In [ ]:
tbl_ma = bucket_validation(lab_ma, MA_STATES, "Moving-average regime (SMA50 / SMA200) and the next 5-day return")

In [ ]:
tbl_ta = bucket_validation(lab_ta, TA_STATES, "Combined TA score (6 rule-based signals) and the next 5-day return")

### Which features does the model rely on?

A single Random Forest is trained on the first 70% of dates (pooled across all stocks, each stock's target scaled by its own volatility) and **permutation importance** is measured on the last 30%: how much the error grows when one feature is shuffled. Values near zero mean the feature adds nothing out of sample.

In [ ]:
pool = pd.concat([datasets[t][0].join(datasets[t][1]).dropna().assign(t=t) for t in datasets]).sort_index()
dates_u = pool.index.unique().sort_values()
cut = dates_u[int(len(dates_u) * 0.7)]
train, test = pool[pool.index < cut - pd.Timedelta(days=10)], pool[pool.index >= cut]     # 10-day purge gap
sd = train.groupby("t")["y"].std()
y_tr = (train["y"] / train["t"].map(sd)).clip(-4, 4)
y_te = test["y"] / test["t"].map(sd)

rf = RandomForestRegressor(n_estimators=150, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(train[ALL_COLS], y_tr)
pi = permutation_importance(rf, test[ALL_COLS], y_te, n_repeats=3, random_state=RANDOM_STATE,
                            scoring="neg_mean_squared_error", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=ALL_COLS)
print(f"Pooled out-of-sample R2 of this Random Forest: {rf.score(test[ALL_COLS], y_te):.4f}  (about 0 or below = little predictability)")

def feat_group(c):
    if c.startswith("w_") or c == "rsi_d_minus_w":
        return "Weekly RSI / MA / MACD"
    if c.startswith("mkt_"):
        return "Market (SPY)"
    if c.startswith("ret_") or c == "dist_high":
        return "Returns / momentum"
    if c.startswith("vol_") or c in ("atr14", "bb_width", "obv20"):
        return "Volatility / volume"
    if c.startswith("sma") or c == "ema12_26":
        return "Moving averages"
    if c in ("rsi14", "rsi_chg5", "stoch_k", "cci20"):
        return "Daily RSI & oscillators"
    return "MACD / Bollinger / trend strength"

grp = imp.groupby(imp.index.map(feat_group)).sum().sort_values()
top = imp.sort_values().tail(15)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].barh(top.index, top.values, color=C["blue"], height=0.65)
ax[0].axvline(0, color=AXIS, lw=1); ax[0].set_title("Top 15 features (increase in error when shuffled)"); ax[0].grid(axis="y", visible=False)
ax[1].barh(grp.index, grp.values, color=C["blue"], height=0.6)
ax[1].axvline(0, color=AXIS, lw=1); ax[1].set_title("Importance by indicator family"); ax[1].grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### Do the new indicators improve the forecast? (ablation)

The same walk-forward test, on the **same dates**, with three feature sets. If adding moving averages, daily TA and weekly RSI does not lower the error versus naive or raise the rank correlation, they are decoration rather than signal.

In [ ]:
def quick_eval(res_dict, model):
    rows = []
    for t, r in res_dict.items():
        ev = r.dropna(subset=["y"])
        ic = stats.spearmanr(ev[model], ev["y"])[0] if ev[model].std() > 0 else np.nan
        rows.append(dict(rmse_ratio=rmse(ev["y"], ev[model]) / rmse(ev["y"], ev["Naive"]),
                         dir_edge=((ev[model] > 0) == (ev["y"] > 0)).mean() - (ev["y"] > 0).mean(), ic=ic))
    d = pd.DataFrame(rows)
    return {"mean_rmse_vs_naive": d["rmse_ratio"].mean(), "pct_beat_naive": (d["rmse_ratio"] < 1).mean() * 100,
            "dir_edge_pct_pts": d["dir_edge"].mean() * 100, "mean_ic": d["ic"].mean()}

if RUN_ABLATION:
    sets = {"A. Base (22 features)": BASE_COLS,
            "B. + moving averages & daily TA (32)": BASE_COLS + TA_COLS,
            "C. + weekly RSI / MA / MACD (38, full)": ALL_COLS}
    abl = {}
    for name, cols in sets.items():
        if cols is ALL_COLS:
            abl[name] = results                              # already computed in section 5
            continue
        o = Parallel(n_jobs=-1)(delayed(walk_forward)(datasets[t][0], datasets[t][1], datasets[t][2], cols, ("Ridge", "HistGB"))
                                for t in tickers)
        abl[name] = dict(zip(tickers, o))
    rows = [dict(feature_set=name, model=m, **quick_eval(res_d, m)) for name, res_d in abl.items() for m in ("Ridge", "HistGB")]
    abl_tbl = pd.DataFrame(rows).set_index(["feature_set", "model"])
    display(abl_tbl.round(4))
    print("Lower mean_rmse_vs_naive is better (below 1 beats the naive forecast); higher dir_edge and mean_ic are better.")
else:
    print("Ablation skipped (RUN_ABLATION = False).")

## 9. Technical dashboard: daily and weekly RSI, moving averages, trend

For every stock: RSI on both time frames (weekly is **live**, including the week in progress), price against its moving averages, golden/death cross state, MACD direction on both frames, and the combined TA score.

In [ ]:
def zone(v):
    if not np.isfinite(v):
        return "n/a"
    return "overbought" if v >= RSI_OB else ("oversold" if v <= RSI_OS else "neutral")

def ta_label_from_score(s):
    return "Bullish" if s >= 3 else ("Bearish" if s <= -3 else "Neutral")

snap_rows = []
for t, p in panels.items():
    last = p.iloc[-1]
    wk_live = weekly_panel(stocks[t]).iloc[-1]
    sg = np.sign(p["sma50"] - p["sma200"]).dropna()
    state, since = "n/a", np.nan
    if len(sg):
        state = "golden (50>200)" if sg.iloc[-1] > 0 else "death (50<200)"
        changes = sg[sg.ne(sg.shift())]
        if len(changes) > 1:                                 # a cross happened inside the 3-year window
            since = len(sg) - sg.index.get_loc(changes.index[-1])
    snap_rows.append(dict(
        ticker=t, as_of=p.index[-1].date(), close=last["close"],
        rsi_d=last["rsi14"], rsi_d_zone=zone(last["rsi14"]),
        rsi_w=wk_live["w_rsi14"], rsi_w_zone=zone(wk_live["w_rsi14"]), rsi_w_last_full_wk=last["w_rsi14"],
        vs_sma20_pct=(last["close"] / last["sma20"] - 1) * 100, vs_sma50_pct=(last["close"] / last["sma50"] - 1) * 100,
        vs_sma200_pct=(last["close"] / last["sma200"] - 1) * 100,
        ma_cross=state, days_since_cross=since,
        macd_daily="bullish" if last["macd_hist"] > 0 else "bearish",
        macd_weekly="bullish" if last["w_macd_hist"] > 0 else ("bearish" if last["w_macd_hist"] <= 0 else "n/a"),
        bb_pctb=last["bb_pctb"], stoch_k=last["stoch_k"], adx14=last["adx14"], cci20=last["cci20"],
        ta_score=int(last["ta_score"]), ta_label=ta_label_from_score(last["ta_score"])))
snap = pd.DataFrame(snap_rows).set_index("ticker")
display(snap.sort_values("rsi_d").round(1))

both_os = snap[(snap.rsi_d <= RSI_OS) & (snap.rsi_w <= RSI_OS)].index.tolist()
both_ob = snap[(snap.rsi_d >= RSI_OB) & (snap.rsi_w >= RSI_OB)].index.tolist()
dip = snap[(snap.rsi_d <= RSI_OS) & (snap.rsi_w >= 50)].index.tolist()
print("Oversold on BOTH daily and weekly RSI :", both_os or "none")
print("Overbought on BOTH daily and weekly RSI:", both_ob or "none")
print("Daily oversold inside a rising weekly trend (weekly RSI >= 50):", dip or "none")
print("Bullish TA score :", snap[snap.ta_label == "Bullish"].index.tolist())
print("Bearish TA score :", snap[snap.ta_label == "Bearish"].index.tolist())

In [ ]:
# RSI map: weekly RSI (x) vs daily RSI (y), one dot per stock
fig, ax = plt.subplots(figsize=(8.5, 7))
ax.axhspan(RSI_OB, 100, color=AXIS, alpha=0.25, lw=0); ax.axhspan(0, RSI_OS, color=AXIS, alpha=0.25, lw=0)
ax.axvspan(RSI_OB, 100, color=AXIS, alpha=0.25, lw=0); ax.axvspan(0, RSI_OS, color=AXIS, alpha=0.25, lw=0)
for v in (RSI_OS, 50, RSI_OB):
    ax.axhline(v, color=MUTED, lw=0.8, ls="--" if v != 50 else ":"); ax.axvline(v, color=MUTED, lw=0.8, ls="--" if v != 50 else ":")
ax.scatter(snap["rsi_w"], snap["rsi_d"], s=42, color=C["blue"], edgecolor=SURFACE, linewidth=1.2, zorder=3)
for t, r in snap.iterrows():
    if r.rsi_d >= RSI_OB or r.rsi_d <= RSI_OS or r.rsi_w >= RSI_OB or r.rsi_w <= RSI_OS:
        ax.annotate(t, (r.rsi_w, r.rsi_d), textcoords="offset points", xytext=(5, 4), fontsize=8, color=INK2)
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
ax.set_xlabel("Weekly RSI(14), live"); ax.set_ylabel("Daily RSI(14)")
ax.text(97, 97, "overbought on both", ha="right", va="top", color=MUTED, fontsize=9)
ax.text(3, 3, "oversold on both", ha="left", va="bottom", color=MUTED, fontsize=9)
ax.set_title("RSI map: daily vs weekly (labelled = in an extreme zone)")
plt.show()

In [ ]:
def ta_chart(t, days=420):
    p = panels[t].iloc[-days:]
    wk = weekly_panel(stocks[t])
    wk = wk[wk.index >= p.index[0]].copy()
    last_day = p.index[-1]
    wk.index = pd.DatetimeIndex([min(i, last_day) for i in wk.index])          # live week plotted at the last trading day
    s = snap.loc[t]
    fig, ax = plt.subplots(4, 1, figsize=(12, 11.5), sharex=True, constrained_layout=True,
                           gridspec_kw={"height_ratios": [3.2, 1.2, 1.2, 1.3]})
    fig.suptitle(f"{t}   daily RSI {s.rsi_d:.0f} ({s.rsi_d_zone})   weekly RSI {s.rsi_w:.0f} ({s.rsi_w_zone})   "
                 f"TA score {s.ta_score:+d} ({s.ta_label})   {s.ma_cross}", x=0.01, ha="left", fontsize=12, fontweight="bold")
    # 1. price, moving averages, Bollinger bands, crosses
    a = ax[0]
    a.fill_between(p.index, p["bb_lo"], p["bb_up"], color=C["blue"], alpha=0.08, linewidth=0)
    a.plot(p.index, p["close"], color=INK, lw=1.3, label="Close")
    a.plot(p.index, p["sma20"], color=C["blue"], lw=1.2, label="SMA 20")
    a.plot(p.index, p["sma50"], color=C["orange"], lw=1.2, label="SMA 50")
    a.plot(p.index, p["sma200"], color=C["aqua"], lw=1.6, label="SMA 200")
    cross = np.sign(p["sma50"] - p["sma200"])
    chg = cross.ne(cross.shift()) & cross.notna() & cross.shift().notna()
    for dt, sgv in cross[chg].items():
        a.scatter([dt], [p.loc[dt, "close"]], marker="^" if sgv > 0 else "v", s=80, color=C["green"] if sgv > 0 else C["red"],
                  edgecolor=SURFACE, linewidth=1.2, zorder=5)
    handles, labels = a.get_legend_handles_labels()
    handles += [Line2D([], [], marker="^", ls="", color=C["green"]), Line2D([], [], marker="v", ls="", color=C["red"])]
    labels += ["Golden cross (50 over 200)", "Death cross (50 under 200)"]
    a.legend(handles, labels, ncol=6, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8.5, columnspacing=1.2, handlelength=1.6)
    a.set_title("Price with moving averages and Bollinger bands (20, 2)", pad=26)
    # 2. daily RSI
    a = ax[1]
    a.axhspan(RSI_OB, 100, color=AXIS, alpha=0.25, lw=0); a.axhspan(0, RSI_OS, color=AXIS, alpha=0.25, lw=0)
    a.plot(p.index, p["rsi14"], color=C["blue"], lw=1.3)
    for v in (RSI_OS, RSI_OB):
        a.axhline(v, color=MUTED, lw=0.8, ls="--")
    a.set_ylim(0, 100); a.set_title("Daily RSI(14)")
    a.annotate(f"{s.rsi_d:.0f}", (last_day, s.rsi_d), textcoords="offset points", xytext=(4, 0), fontsize=9, color=INK2, va="center")
    # 3. weekly RSI
    a = ax[2]
    a.axhspan(RSI_OB, 100, color=AXIS, alpha=0.25, lw=0); a.axhspan(0, RSI_OS, color=AXIS, alpha=0.25, lw=0)
    a.plot(wk.index, wk["w_rsi14"], color=C["orange"], lw=1.5, marker="o", ms=3)
    for v in (RSI_OS, RSI_OB):
        a.axhline(v, color=MUTED, lw=0.8, ls="--")
    a.set_ylim(0, 100); a.set_title("Weekly RSI(14); last point = week in progress")
    a.annotate(f"{s.rsi_w:.0f}", (last_day, s.rsi_w), textcoords="offset points", xytext=(4, 0), fontsize=9, color=INK2, va="center")
    # 4. daily MACD
    a = ax[3]
    a.bar(p.index, p["macd_hist"], width=1.0, color=np.where(p["macd_hist"] >= 0, C["blue"], C["red"]), alpha=0.65, label="Histogram")
    a.plot(p.index, p["macd"], color=INK, lw=1.1, label="MACD")
    a.plot(p.index, p["macd_sig"], color=C["orange"], lw=1.1, label="Signal")
    a.axhline(0, color=AXIS, lw=0.8)
    a.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=8.5)
    a.set_title("Daily MACD (12, 26, 9)", pad=22)
    a.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.show()

for t in [t for t in CHART_TICKERS if t in panels]:
    ta_chart(t)

## 10. Final forecast: ML + technical analysis + fundamentals

The best ML approach is **refit on all 3 years** per stock and forecasts the next `HORIZON` trading days. A stock is flagged **BUY** only if:

1. the forecast return exceeds `BUY_THRESHOLD`, **and**
2. the model showed a **validated edge on that stock** out of sample: it beat the naive forecast (RMSE ratio below 1), beat always-up on direction, and had a positive IC.

Stocks with a bullish forecast but no validated edge are shown as **WATCH**.

**Confluence (0-3)** counts how many independent views agree: ML bullish *with* validated edge, TA score "Bullish", fundamental score at or above `MIN_FUND_SCORE`. The RSI zones are shown as a caution, not scored: a high-confluence stock with an overbought daily and weekly RSI is statistically stretched.

In [ ]:
report_set = set(US_HALAL)
rows = []
for t, (feat, y, c) in datasets.items():
    data = feat.join(y).dropna(subset=list(feat.columns))
    train = data.dropna(subset=["y"])
    Xnow = data[ALL_COLS].iloc[[-1]]
    pred = fit_predict_all(train[ALL_COLS], train["y"], Xnow)[BEST][0]
    ev = results[t].dropna(subset=["y"])
    resid = ev["y"] - ev[BEST]
    lo, hi = np.quantile(resid, [0.10, 0.90])
    m = metrics[(metrics.ticker == t) & (metrics.model == BEST)].iloc[0]
    edge = bool(m["rmse_vs_naive"] < 1 and m["dir_acc"] > m["always_up_acc"] and m["ic"] > 0)
    last = float(c.iloc[-1])
    if pred > BUY_THRESHOLD and edge:
        sig = "BUY"
    elif pred > BUY_THRESHOLD:
        sig = "WATCH (no validated edge)"
    else:
        sig = "NO BUY"
    fs = fund["fund_score"].get(t, np.nan)
    ta_lab = snap.loc[t, "ta_label"]
    confluence = int(sig == "BUY") + int(ta_lab == "Bullish") + int(np.isfinite(fs) and fs >= MIN_FUND_SCORE)
    rows.append({"ticker": t, "in_report": "yes" if t in report_set else "no - verify halal status",
                 "as_of": c.index[-1].date(), "last_close": round(last, 2),
                 f"pred_{HORIZON}d_return_%": round(pred * 100, 2),
                 f"pred_price_{HORIZON}d": round(last * np.exp(pred), 2),
                 "80%_range_low": round(last * np.exp(pred + lo), 2), "80%_range_high": round(last * np.exp(pred + hi), 2),
                 "rmse_vs_naive": round(m["rmse_vs_naive"], 3), "dir_acc_%": round(m["dir_acc"] * 100, 1),
                 "always_up_%": round(m["always_up_acc"] * 100, 1), "IC": round(m["ic"], 3),
                 "validated_edge": edge, "signal": sig,
                 "rsi_daily": round(snap.loc[t, "rsi_d"], 1), "rsi_weekly": round(snap.loc[t, "rsi_w"], 1),
                 "rsi_daily_zone": snap.loc[t, "rsi_d_zone"], "rsi_weekly_zone": snap.loc[t, "rsi_w_zone"],
                 "ta_label": ta_lab, "ta_score": int(snap.loc[t, "ta_score"]),
                 "fund_score": round(fs, 1) if np.isfinite(fs) else np.nan, "fund_view": fund["fund_view"].get(t, np.nan),
                 "pe": fund["pe"].get(t, np.nan), "roe": fund["roe"].get(t, np.nan), "de": fund["de"].get(t, np.nan),
                 "current_ratio": fund["current"].get(t, np.nan), "confluence": confluence})

signals = (pd.DataFrame(rows)
           .sort_values(["confluence", f"pred_{HORIZON}d_return_%"], ascending=[False, False]).reset_index(drop=True))
buys = signals[signals["signal"] == "BUY"]
print(f"Model: {BEST} | horizon: {HORIZON} trading days | as of latest close in Yahoo data")
print(f"BUY: {len(buys)}   WATCH: {(signals.signal.str.startswith('WATCH')).sum()}   NO BUY: {(signals.signal == 'NO BUY').sum()}\n")

cols_show = ["ticker", "in_report", "last_close", f"pred_{HORIZON}d_return_%", f"pred_price_{HORIZON}d", "80%_range_low", "80%_range_high",
             "rsi_daily", "rsi_weekly", "ta_label", "fund_score", "fund_view", "confluence", "signal"]
print("ML BUY list (validated edge + bullish forecast):")
if len(buys):
    display(buys[cols_show])
else:
    print("No stock passed both tests (bullish forecast AND validated out-of-sample edge). That is a legitimate result: "
          "it means the models did not find reliable predictability at this horizon.")

full = signals[signals["confluence"] == 3]
print("\nAll three views agree (ML BUY + TA Bullish + fundamentals OK):")
display(full[cols_show]) if len(full) else print("none")
print("\nTop 15 by confluence, then predicted return (whatever the ML signal):")
display(signals[cols_show].head(15))

In [ ]:
print("Bullish forecasts without a validated edge (treat with suspicion):")
display(signals[signals.signal.str.startswith("WATCH")][cols_show])

signals.to_csv("signals.csv", index=False)
metrics.to_csv("validation_metrics.csv", index=False)
snap.to_csv("technical_snapshot.csv")
fund.to_csv("fundamentals.csv")
try:
    from google.colab import files
    for f_ in ("signals.csv", "validation_metrics.csv", "technical_snapshot.csv", "fundamentals.csv"):
        files.download(f_)
except Exception:
    print("Saved signals.csv, validation_metrics.csv, technical_snapshot.csv and fundamentals.csv in the working directory.")

## How to read the results honestly

- **Markets are close to efficient.** On daily/weekly data, ML models rarely beat "price stays the same" by much, and a few stocks will look skilled by luck. That is why the notebook uses non-overlapping tests, BH correction, benchmarks, an ablation and a cost-aware backtest.
- **RSI and moving averages are famous but not magic.** Section 8 tests them directly. If the buckets show small effects, p-values above ~0.05, or opposite signs between the stock-level and date-level tests, then "RSI above 70 means sell" is a habit, not an edge in this sample.
- **Only 3 years** (about 750 days; Sandisk has less) means few independent 5-day observations per stock. The 200-day average and the weekly MACD need long warm-ups, so the TA-score and MA-regime tests use only the later ~550 days. Trust the *pooled* tables more than any single ticker.
- **Fundamentals are a snapshot**, not history, so they cannot be back-tested here. Yahoo's ratios can also be stale or missing (e.g. no debt reported). The fundamental score is a transparent rule of thumb, not a valuation model.
- **A BUY is a statistical tilt, not a promise.** The 80% range columns show how wide the realistic outcome band is, usually far wider than the predicted move.
- **Regime risk:** 2023-2026 was dominated by an AI/semiconductor rally. A model trained on it may simply have learned "momentum in chips works". Check the quarterly stability table.
- **Halal status:** only the 50 names from the report were screened by that report's methodology. SK Hynix, Rambus and the other cybersecurity additions were *not* screened here; verify them with your own scholar/screen before buying. Book debt-to-equity here is not the AAOIFI debt-to-market-cap test.
- To change the horizon (e.g. 1 day or 21 days), edit `HORIZON` in the config cell and re-run everything.

*Not investment advice.*